# AI Research Paper Validator -- v0 (Kaggle/Colab runner)

Reads a research paper PDF, extracts its headline claim + code repo, clones and runs the repo's evaluation, and checks whether the claimed metric reproduces.

**All model compute and repo execution happens in THIS notebook's runtime (Kaggle/Colab GPU), not on the author's laptop.**

Setup:
1. On Kaggle: enable **Internet** and a **GPU** accelerator (Settings panel, right sidebar). On Colab: `Runtime > Change runtime type > GPU`.
2. Run every cell top to bottom.
3. In the "Provide input PDF" cell, either upload a file (Colab) or point `PDF_PATH` at a Kaggle input dataset file.

This notebook is self-contained: it writes out the `src/` pipeline package via `%%writefile` so it doesn't depend on this repo being pushed anywhere.

## 1. Install dependencies

llama-cpp-python is built with CUDA support so inference offloads to the GPU.

In [ ]:
import subprocess

!pip install -q --upgrade pip
!pip install -q huggingface_hub PyMuPDF

# Try a prebuilt CUDA wheel first (fast, no compilation). If the CUDA
# version on this image doesn't match, fall back to the prebuilt
# CPU-only wheel from PyPI instead of silently compiling llama.cpp
# from source (which can take 10-20+ minutes and looks like a hang).
cuda_install = subprocess.run(
    ["pip", "install", "-q", "llama-cpp-python",
     "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121"],
    capture_output=True, text=True,
)
if cuda_install.returncode != 0:
    print("CUDA wheel install failed, falling back to CPU-only build:")
    print(cuda_install.stderr[-2000:])
    subprocess.run(["pip", "install", "-q", "llama-cpp-python"], check=True)
    print("Installed llama-cpp-python (CPU-only -- inference will be slower)")
else:
    print("Installed llama-cpp-python (CUDA build)")


If the prebuilt CUDA wheel above fails to install for your CUDA version, fall back to a plain CPU build (slower but works):

```python
!CMAKE_ARGS="-DGGML_CUDA=off" pip install -q llama-cpp-python
```

## 2. Download the local model

Qwen2.5-3B-Instruct, quantized (Q4_K_M GGUF, ~2GB). Swap `MODEL_REPO`/`MODEL_FILE` for a bigger or smaller quant if you want to trade quality for speed.

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REPO = "Qwen/Qwen2.5-3B-Instruct-GGUF"
MODEL_FILE = "qwen2.5-3b-instruct-q4_k_m.gguf"

model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
print("Model downloaded to:", model_path)


## 3. Write out the pipeline package

This mirrors `buildathon-claude/src/` from the project repo exactly -- edit there and re-copy these cells if you change the pipeline logic.

In [ ]:
import os
os.makedirs("src", exist_ok=True)
open("src/__init__.py", "w").close()
print("src/__init__.py created")


In [ ]:
%%writefile src/config.py
from dataclasses import dataclass


@dataclass
class Config:
    max_runtime_retries: int = 5
    log_truncate_lines: int = 30
    subprocess_timeout_sec: int = 900
    metric_tolerance_relative: float = 0.05
    workdir: str = "./runs"


In [ ]:
%%writefile src/llm_client.py
"""
Pluggable LLM backend used by every pipeline stage.

Swap backends via `get_llm_client(backend, ...)`:
  - "mock":      canned responses, no model, no network. Used for local
                 wiring tests on a machine with no GPU (e.g. this laptop).
  - "llama_cpp": local quantized GGUF model (e.g. Qwen2.5-Instruct) run via
                 llama-cpp-python. Intended to run ONLY inside the
                 Kaggle/Colab notebook (notebooks/kaggle_colab_runner.ipynb),
                 where a GPU and enough RAM are available.
  - "anthropic": placeholder for swapping to the Claude API once keys are
                 available. Not implemented in v0.

Every pipeline stage talks to LLMClient.complete() / .complete_json() only,
so changing the backend never touches pipeline logic.
"""
from __future__ import annotations

import json
import re
from abc import ABC, abstractmethod


class LLMClient(ABC):
    @abstractmethod
    def complete(self, system_prompt: str, user_prompt: str, max_tokens: int = 1024) -> str:
        ...

    def complete_json(self, system_prompt: str, user_prompt: str, max_tokens: int = 1024) -> dict:
        raw = self.complete(system_prompt, user_prompt, max_tokens=max_tokens)
        return _parse_json_loose(raw)


def _escape_stray_backslashes(s: str) -> str:
    """Small local models sometimes emit invalid JSON escapes (e.g. a
    hyphenation artifact like 'LAN-\\GUAGE' from PDF text, where \\G isn't
    a valid JSON escape). Double any backslash not already starting a
    legal escape sequence so json.loads can parse it."""
    return re.sub(r'\\(?!["\\/bfnrtu])', r"\\\\", s)


def _parse_json_loose(raw: str) -> dict:
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    candidates = [raw] + ([match.group(0)] if match else [])
    candidates += [_escape_stray_backslashes(c) for c in candidates]
    for candidate in candidates:
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            continue
    raise ValueError(f"Could not parse JSON from LLM response:\n{raw!r}")


class MockLLMClient(LLMClient):
    """Returns pre-queued responses in order. For local, model-free testing
    of pipeline wiring only -- never used for real extraction or eval."""

    def __init__(self, responses: list[str]):
        self._responses = list(responses)
        self.calls: list[tuple[str, str]] = []

    def complete(self, system_prompt: str, user_prompt: str, max_tokens: int = 1024) -> str:
        self.calls.append((system_prompt, user_prompt))
        if not self._responses:
            raise RuntimeError("MockLLMClient ran out of queued responses")
        return self._responses.pop(0)


class LlamaCppLLMClient(LLMClient):
    """Local quantized GGUF model via llama-cpp-python.

    Instantiate this ONLY inside the Kaggle/Colab notebook where
    llama-cpp-python is installed and a GPU is available. llama_cpp is
    imported lazily so merely importing this module elsewhere (e.g. on a
    laptop with no GPU) never pulls it in.
    """

    def __init__(self, model_path: str, n_ctx: int = 8192, n_gpu_layers: int = -1, verbose: bool = False):
        from llama_cpp import Llama  # lazy import: only required on Kaggle/Colab

        self._llm = Llama(
            model_path=model_path,
            n_ctx=n_ctx,
            n_gpu_layers=n_gpu_layers,  # -1 = offload all layers to GPU if available
            verbose=verbose,
        )

    def complete(self, system_prompt: str, user_prompt: str, max_tokens: int = 1024) -> str:
        result = self._llm.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            max_tokens=max_tokens,
            temperature=0.1,
        )
        return result["choices"][0]["message"]["content"]


def get_llm_client(backend: str, **kwargs) -> LLMClient:
    if backend == "mock":
        return MockLLMClient(kwargs.get("responses", []))
    if backend == "llama_cpp":
        return LlamaCppLLMClient(
            model_path=kwargs["model_path"],
            n_ctx=kwargs.get("n_ctx", 8192),
            n_gpu_layers=kwargs.get("n_gpu_layers", -1),
            verbose=kwargs.get("verbose", False),
        )
    raise ValueError(f"Unknown LLM backend: {backend!r}")


In [ ]:
%%writefile src/pdf_extract.py
from __future__ import annotations


def extract_text(pdf_path: str, max_chars: int = 20000) -> str:
    """Pull raw text out of a research paper PDF. v0 is text-only (no VLM),
    so figures and complex tables may be lost -- adequate for pulling repo
    URLs, dataset names, and headline metrics out of the body text."""
    import fitz  # PyMuPDF, lazy import

    doc = fitz.open(pdf_path)
    try:
        text = "\n".join(page.get_text() for page in doc)
    finally:
        doc.close()
    return text[:max_chars]


In [ ]:
%%writefile src/extraction.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

from .llm_client import LLMClient

SYSTEM_PROMPT = """You are an expert research-paper analyst. Given raw text \
extracted from an academic paper, extract the information needed to \
reproduce its headline result. Respond with ONLY a single JSON object, no \
prose, no markdown fences, matching exactly this schema:

{
  "title": string,
  "github_repo": string or null,   // full https URL to the official code repo, if mentioned
  "dataset": string or null,       // dataset name or HuggingFace dataset id used for the headline result
  "claimed_metric_name": string,   // e.g. "accuracy", "F1", "BLEU"
  "claimed_metric_value": number,  // the headline number reported, as a plain float (e.g. 95.1 not "95.1%")
  "claimed_metric_unit": string,   // e.g. "%", "points"
  "eval_notes": string             // 1-3 sentences on which model/dataset/split produced this number, for someone about to reproduce it
}

If a field cannot be determined from the text, use null (or 0 for the metric value, but only if truly absent)."""


@dataclass
class ExtractedClaim:
    title: str
    github_repo: Optional[str]
    dataset: Optional[str]
    claimed_metric_name: str
    claimed_metric_value: float
    claimed_metric_unit: str
    eval_notes: str

    @classmethod
    def from_dict(cls, d: dict) -> "ExtractedClaim":
        return cls(
            title=d.get("title") or "Unknown",
            github_repo=d.get("github_repo") or None,
            dataset=d.get("dataset") or None,
            claimed_metric_name=d.get("claimed_metric_name") or "unknown metric",
            claimed_metric_value=float(d.get("claimed_metric_value") or 0.0),
            claimed_metric_unit=d.get("claimed_metric_unit") or "",
            eval_notes=d.get("eval_notes") or "",
        )


def extract_claim(llm: LLMClient, paper_text: str) -> ExtractedClaim:
    user_prompt = f"Paper text:\n\n{paper_text}"
    data = llm.complete_json(SYSTEM_PROMPT, user_prompt, max_tokens=800)
    return ExtractedClaim.from_dict(data)


In [ ]:
%%writefile src/provisioning.py
from __future__ import annotations

import subprocess
from pathlib import Path


class ProvisioningError(RuntimeError):
    pass


def clone_repo(repo_url: str, dest_dir: Path, timeout_sec: int = 300) -> Path:
    dest_dir.parent.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ["git", "clone", "--depth", "1", repo_url, str(dest_dir)],
        capture_output=True, text=True, timeout=timeout_sec,
    )
    if result.returncode != 0:
        raise ProvisioningError(f"git clone failed:\n{result.stderr}")
    return dest_dir


def install_requirements(repo_dir: Path, timeout_sec: int = 600) -> str:
    """Best-effort dependency install from requirements.txt. Returns combined
    stdout+stderr for logging. No-op (returns "") if no requirements.txt."""
    req_file = repo_dir / "requirements.txt"
    if not req_file.exists():
        return ""
    result = subprocess.run(
        ["pip", "install", "-q", "-r", str(req_file)],
        capture_output=True, text=True, timeout=timeout_sec,
    )
    log = result.stdout + result.stderr
    if result.returncode != 0:
        raise ProvisioningError(log)
    return log


In [ ]:
%%writefile src/execution.py
from __future__ import annotations

import re
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

from .llm_client import LLMClient

SYSTEM_PROMPT = """You are helping reproduce a research paper's result inside \
a Python environment where the paper's official code repository has already \
been cloned and its requirements.txt installed. Given a partial file listing \
of the repo and its README, propose how to run its evaluation.

Respond with ONLY a single JSON object, no prose:
{
  "command": string,       // a single shell command to run from the repo root that runs evaluation and prints a final metric to stdout
  "metric_regex": string   // a Python regex with exactly one capture group that extracts the final numeric metric from that command's stdout
}"""


@dataclass
class RunPlan:
    command: str
    metric_regex: str


def propose_run_plan(llm: LLMClient, repo_dir: Path, dataset: Optional[str]) -> RunPlan:
    listing = "\n".join(
        str(p.relative_to(repo_dir))
        for p in sorted(repo_dir.rglob("*"))
        if p.is_file() and ".git" not in p.parts
    )[:4000]
    readme = ""
    for name in ("README.md", "README.rst", "README.txt"):
        f = repo_dir / name
        if f.exists():
            readme = f.read_text(errors="ignore")[:4000]
            break
    user_prompt = (
        f"Dataset used for the headline result: {dataset or 'unknown'}\n\n"
        f"Repo file listing:\n{listing}\n\nREADME:\n{readme}"
    )
    data = llm.complete_json(SYSTEM_PROMPT, user_prompt, max_tokens=500)
    return RunPlan(command=data["command"], metric_regex=data["metric_regex"])


@dataclass
class RunResult:
    returncode: int
    stdout: str
    stderr: str


def run_command(command: str, cwd: Path, timeout_sec: int = 900) -> RunResult:
    try:
        result = subprocess.run(
            command, shell=True, cwd=str(cwd),
            capture_output=True, text=True, timeout=timeout_sec,
        )
        return RunResult(result.returncode, result.stdout, result.stderr)
    except subprocess.TimeoutExpired as e:
        stdout = e.stdout if isinstance(e.stdout, str) else (e.stdout or b"").decode(errors="ignore")
        stderr = e.stderr if isinstance(e.stderr, str) else (e.stderr or b"").decode(errors="ignore")
        return RunResult(-1, stdout, stderr + "\n[TIMED OUT]")


def extract_metric(stdout: str, metric_regex: str) -> Optional[float]:
    match = re.search(metric_regex, stdout)
    if not match:
        return None
    try:
        return float(match.group(1))
    except (ValueError, IndexError):
        return None


def truncate_log(text: str, max_lines: int = 30) -> str:
    lines = text.strip().splitlines()
    return "\n".join(lines[-max_lines:])


In [ ]:
%%writefile src/debug_loop.py
from __future__ import annotations

from pathlib import Path

from .execution import RunResult, run_command, truncate_log
from .llm_client import LLMClient

SYSTEM_PROMPT = """You are debugging a failed attempt to run a research \
paper's evaluation script. You will see the failing command and the tail of \
its error output. Respond with ONLY a single JSON object, no prose:

{
  "patch_command": string,  // a single shell command that fixes the problem (e.g. "pip install foo==1.2.3"), to run before retrying
  "reasoning": string       // one sentence on what went wrong
}

If you cannot tell what would fix it, set "patch_command" to an empty string."""


def attempt_patch(llm: LLMClient, failing_command: str, stderr_tail: str, repo_dir: Path) -> str:
    user_prompt = f"Failing command:\n{failing_command}\n\nError output (tail):\n{stderr_tail}"
    data = llm.complete_json(SYSTEM_PROMPT, user_prompt, max_tokens=300)
    patch_command = data.get("patch_command") or ""
    if patch_command:
        run_command(patch_command, cwd=repo_dir, timeout_sec=300)
    return patch_command


def run_with_debug_loop(
    llm: LLMClient,
    command: str,
    repo_dir: Path,
    max_retries: int,
    truncate_lines: int,
    timeout_sec: int,
) -> RunResult:
    result = run_command(command, cwd=repo_dir, timeout_sec=timeout_sec)
    attempts = 0
    while result.returncode != 0 and attempts < max_retries:
        stderr_tail = truncate_log(result.stderr, truncate_lines)
        patch_command = attempt_patch(llm, command, stderr_tail, repo_dir)
        if not patch_command:
            break
        result = run_command(command, cwd=repo_dir, timeout_sec=timeout_sec)
        attempts += 1
    return result


In [ ]:
%%writefile src/verification.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

from .extraction import ExtractedClaim


@dataclass
class VerificationResult:
    claim: ExtractedClaim
    reproduced_value: Optional[float]
    passed: bool
    reason: str

    def to_markdown(self) -> str:
        status = "PASS" if self.passed else "FAIL"
        repro = f"{self.reproduced_value}" if self.reproduced_value is not None else "N/A"
        return f"""# Validation Report: {self.claim.title}

| Field | Value |
|---|---|
| Repo | {self.claim.github_repo or 'unknown'} |
| Dataset | {self.claim.dataset or 'unknown'} |
| Claimed metric | {self.claim.claimed_metric_name} = {self.claim.claimed_metric_value}{self.claim.claimed_metric_unit} |
| Reproduced metric | {repro}{self.claim.claimed_metric_unit} |
| **Result** | **{status}** |

**Notes:** {self.reason}

_{self.claim.eval_notes}_
"""


def verify(claim: ExtractedClaim, reproduced_value: Optional[float], tolerance_relative: float) -> VerificationResult:
    if reproduced_value is None:
        return VerificationResult(claim, None, False, "Could not extract a reproduced metric from the eval output.")
    claimed = claim.claimed_metric_value
    if claimed == 0:
        return VerificationResult(claim, reproduced_value, False, "Claimed metric value unknown; cannot compare.")
    relative_diff = abs(reproduced_value - claimed) / abs(claimed)
    passed = relative_diff <= tolerance_relative
    reason = f"Reproduced value differs from claimed by {relative_diff:.1%} (tolerance: {tolerance_relative:.0%})."
    return VerificationResult(claim, reproduced_value, passed, reason)


In [ ]:
%%writefile src/pipeline.py
from __future__ import annotations

from pathlib import Path

from .config import Config
from .debug_loop import run_with_debug_loop
from .execution import extract_metric, propose_run_plan, truncate_log
from .extraction import extract_claim
from .llm_client import LLMClient
from .pdf_extract import extract_text
from .provisioning import ProvisioningError, clone_repo, install_requirements
from .verification import VerificationResult, verify


def run_pipeline(pdf_path: str, llm: LLMClient, config: Config) -> VerificationResult:
    paper_text = extract_text(pdf_path)
    claim = extract_claim(llm, paper_text)

    if not claim.github_repo:
        return VerificationResult(claim, None, False, "No GitHub repo found in the paper; cannot reproduce.")

    workdir = Path(config.workdir) / Path(pdf_path).stem
    repo_dir = workdir / "repo"

    try:
        clone_repo(claim.github_repo, repo_dir)
        install_requirements(repo_dir)
    except ProvisioningError as e:
        return VerificationResult(claim, None, False, f"Provisioning failed: {e}")

    plan = propose_run_plan(llm, repo_dir, claim.dataset)
    result = run_with_debug_loop(
        llm, plan.command, repo_dir,
        max_retries=config.max_runtime_retries,
        truncate_lines=config.log_truncate_lines,
        timeout_sec=config.subprocess_timeout_sec,
    )

    if result.returncode != 0:
        stderr_tail = truncate_log(result.stderr, config.log_truncate_lines)
        reason = (
            f"Evaluation script failed after retries.\n\n"
            f"Command tried: `{plan.command}`\n\n"
            f"Stderr (tail):\n```\n{stderr_tail}\n```"
        )
        return VerificationResult(claim, None, False, reason)

    reproduced_value = extract_metric(result.stdout, plan.metric_regex)
    return verify(claim, reproduced_value, config.metric_tolerance_relative)


## 4. Provide the input PDF

Colab: uploads a file via the widget below. Kaggle: comment out the Colab block and set `PDF_PATH` to a file under `/kaggle/input/...` (add the PDF as a Kaggle dataset first).

In [ ]:
PDF_PATH = None

try:
    from google.colab import files
    uploaded = files.upload()
    PDF_PATH = next(iter(uploaded.keys()))
except ImportError:
    # Running on Kaggle (or elsewhere): point this at your PDF instead.
    PDF_PATH = "/kaggle/input/your-dataset/paper.pdf"

print("Using PDF:", PDF_PATH)


## 5. Run the pipeline

In [ ]:
import sys
sys.path.insert(0, ".")

from src.config import Config
from src.llm_client import get_llm_client
from src.pipeline import run_pipeline

llm = get_llm_client("llama_cpp", model_path=model_path, n_gpu_layers=-1)
config = Config()

result = run_pipeline(PDF_PATH, llm, config)
report = result.to_markdown()
print(report)

with open("report.md", "w") as f:
    f.write(report)


## Known v0 limitations

- Text-only PDF extraction (no vision model) -- tables/figures may be missed.
- The LLM proposes the eval *command* and a regex to pull the metric out of stdout; for repos with unusual eval entrypoints this guess can be wrong (rerun, or manually set `plan.command` after inspecting `src/execution.py`'s `propose_run_plan`).
- Dependency install is a single best-effort `pip install -r requirements.txt`; only the *run* step (not the install step) goes through the debug/patch loop.
- No sandboxing beyond the Kaggle/Colab VM itself -- fine for a personal MVP against trusted repos, not for untrusted code. Swapping in Modal (`modal.Sandbox`) is the natural v1 step, together with swapping this local Qwen model for the Anthropic API -- both are isolated behind `src/llm_client.py`'s backend interface, so neither swap should require touching `src/pipeline.py`.